# Agentic segmentation pipeline

Grounding DINO → MedSAM loop reviewed by Gemma 4 E4B (multimodal, cloud).

Before running: set `GEMINI_API_KEY` in Colab Secrets (🔑 left pane).

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()

In [ ]:
# === Load models (cached across cell reruns) ========================
from pipeline.models import get_dino_model, get_medsam_model, get_gemma_client

_ = get_dino_model()
_ = get_medsam_model()
gemma = get_gemma_client()
print('models ready')

In [ ]:
# === Gemma smoke test (text + multimodal round-trip) ================
import json
from pathlib import Path
from pipeline import config, medsam
from pipeline.gemma import smoke_test

sample_img = next(config.KVASIR_IMAGES.glob('*.jpg'))
image_np = medsam.load_image(str(sample_img))
print(json.dumps(smoke_test(gemma, image_np), indent=2))

In [ ]:
# === Single-image demo ==============================================
import matplotlib.pyplot as plt
import numpy as np
from skimage import io
from pipeline import config
from pipeline.agent import run_agent

demo_file = next(config.KVASIR_IMAGES.glob('*.jpg'))
gt_path = config.KVASIR_MASKS / demo_file.name
gt = None
if gt_path.exists():
    m = io.imread(gt_path)
    gt = (m[..., 0] if m.ndim == 3 else m) > 0
    gt = gt.astype(np.uint8)

state = run_agent(str(demo_file), 'find the polyp', gt_mask=gt)
print(f'run_id         : {state.run_id}')
print(f'refined prompt : {state.refined_prompt}')
print(f'stop reason    : {state.stop_reason}')
for it in state.iterations:
    act = it.gemma_action.action if it.gemma_action else '—'
    print(f'  iter {it.iteration}: action={act:<14} '
          f'dice={it.dice_vs_gt} area_pct={it.metrics.area_pct:.3f}')

# Plot original + each iteration's mask overlay
n = 1 + len(state.iterations)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
axes[0].imshow(state.image_np); axes[0].set_title('image'); axes[0].axis('off')
for i, it in enumerate(state.iterations, start=1):
    axes[i].imshow(state.image_np)
    axes[i].imshow(it.mask, alpha=0.45, cmap='Reds')
    axes[i].set_title(f'iter {it.iteration}'
                      + (f'  DICE={it.dice_vs_gt:.2f}' if it.dice_vs_gt is not None else ''))
    axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# === Batch evaluation on test.json ==================================
from pipeline.eval import run_batch

df = run_batch(n=20)  # set n=None for the full test set
df.describe()[['first_iter_dice', 'final_dice', 'n_iterations']]

In [ ]:
# === One-shot vs agentic comparison =================================
from pipeline.eval import compare_oneshot_vs_agentic

cmp_df = compare_oneshot_vs_agentic(n=50)
print(cmp_df[['one_shot_dice', 'agentic_dice', 'dice_delta']].describe())